# 🛩️ NASA Turbofan Jet Engine - Preprocessing + Modeling (No EDA, No Charts)

- Giữ nguyên pipeline xử lý dữ liệu của notebook cũ
- Bỏ toàn bộ phần EDA/biểu đồ
- Chỉ train 2 model: RandomForest và XGBoost
- Chọn tập dữ liệu cần chạy bằng biến `TARGET_FDS`


## 1. Setup & Load Data

In [827]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')
print('Libraries loaded successfully.')


Libraries loaded successfully.


In [ ]:
# Define column names
col_names = ['unit_id', 'cycle'] + \
            [f'op_setting_{i}' for i in range(1, 4)] + \
            [f'sensor_{i}' for i in range(1, 22)]

DATA_DIR = 'CMaps'
TARGET_FDS = ['FD003', 'FD004']  # đổi tại đây nếu muốn chạy bộ khác

# Load all datasets
datasets = {}
for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    # Resolve dataset folder robustly (handles different notebook working directories)
    candidate_dirs = [
        DATA_DIR,
        os.path.join('.', DATA_DIR),
        os.path.join('..', DATA_DIR),
        os.getcwd(),
        os.path.join(os.getcwd(), DATA_DIR),
    ]

    loaded = False
    checked_paths = []
    for base_dir in dict.fromkeys(candidate_dirs):  # remove duplicates, keep order
        train_path = os.path.join(base_dir, f'train_{fd}.txt')
        test_path  = os.path.join(base_dir, f'test_{fd}.txt')
        rul_path   = os.path.join(base_dir, f'RUL_{fd}.txt')
        checked_paths.append((train_path, test_path, rul_path))

        if all(os.path.isfile(p) for p in [train_path, test_path, rul_path]):
            datasets[fd] = {
                'train': pd.read_csv(train_path, sep=r'\s+', header=None, names=col_names),
                'test':  pd.read_csv(test_path,  sep=r'\s+', header=None, names=col_names),
                'rul':   pd.read_csv(rul_path,   sep=r'\s+', header=None, names=['RUL'])
            }
            loaded = True
            break

    if not loaded:
        raise FileNotFoundError(
            f"Could not find files for {fd}. Current working directory: {os.getcwd()}\n"
            f"Checked:\n" + "\n".join([f"  - {t}\n    {te}\n    {r}" for t, te, r in checked_paths])
        )

print('All 4 sub-datasets (train, test, RUL for each) have been loaded.')
print(f'Columns: {col_names}')


All 4 sub-datasets (train, test, RUL for each) have been loaded.
Columns: ['unit_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


# Data processing

# Giai đoạn 1: Làm sạch mảng (Data Purging) & Khởi tạo Nhãn (Labeling) 

### Task 1.1 + 1.2: Loại bỏ các sensor không có giá trị: std ~ 0 và kiểm tra các sensor có giá trị biến động rất nhỏ

Trong hệ thống của máy bay, 3 cột này đại diện cho các điều kiện môi trường do phi công điều khiển:

op_setting_1: Độ cao bay (Altitude)

op_setting_2: Tốc độ bay (Mach Number)

op_setting_3: Góc mở bướm ga (Throttle Resolver Angle)

Tuy nhiên, ở fd001 và fd003, chỉ có 1 điều kiện bay là so với mực nước biển, trong điều kiện này phi công không thay đổi bướm ga, vì thế std = 0 nên cần được xóa bỏ, và ở op1 và op2 cũng tương tự, không thay đổi độ cao và tốc độ bay, giá trị chỉ nhúc nhích một chút (white noise) nên cũng áp dụng xóa bỏ

Có 2 near - constant sensor ở fd001 là 16 và ở fd003 là 10, sẽ không vội xóa, mà tính ma trận tương quan để xem có ý nghĩa mặt giá trị hay không rồi mới quyết định xóa hay không

In [851]:
target_fds = TARGET_FDS
feature_cols = [f'sensor_{i}' for i in range(1, 22)] + [f'op_setting_{i}' for i in range(1, 4)]

for fd in target_fds:
    print(f"\n{'='*10} [{fd}] TASK 1.1 & 1.2: SENSOR AND FEATURE FILTERING {'='*10}")
    train_df = datasets[fd]['train']
    test_df = datasets[fd]['test']
    
    # Calculate standard deviation of features
    std_vals = train_df[feature_cols].std()
    
    # 1. Group Constant features
    constant_cols = std_vals[std_vals < 1e-6].index.tolist()
    
    # 2. Group Near-constant features for testing
    near_constant_cols = std_vals[(std_vals >= 1e-6) & (std_vals < 0.01)].index.tolist()
    
    # 3. Classify Near-constant features using Correlation Test
    cols_to_drop = constant_cols.copy() # Start drop list with Constant columns
    kept_near_constant = []             # List for saved features
    
    for col in near_constant_cols:
        corr = train_df[col].corr(train_df['cycle'])
        
        # If random fluctuation (low correlation), add to drop list
        if abs(corr) < 0.1 or pd.isna(corr):
            cols_to_drop.append(col)
        # If there is a clear trend, keep it
        else:
            kept_near_constant.append((col, corr))
            
    # 4. Perform drop operation once on both Train and Test sets
    datasets[fd]['train'] = train_df.drop(columns=cols_to_drop)
    datasets[fd]['test'] = test_df.drop(columns=cols_to_drop)
    
    # --- PRINT RESULTS ---
    print(f"Dropped {len(constant_cols)} Constant columns: {constant_cols}")
    
    # Find near-constant columns that were dropped
    dropped_near = [c for c in cols_to_drop if c not in constant_cols]
    print(f"Dropped {len(dropped_near)} Near-constant columns (meaningless noise, corr < 0.1): {dropped_near}")
    
    if kept_near_constant:
        print(f"Successfully KEPT Near-constant columns (with trend):")
        for col, corr in kept_near_constant:
            print(f"    -> {col} (Correlation with cycle: {corr:.4f})")
            
    print(f"Final number of columns retained (including unit_id, cycle, RUL if any): {datasets[fd]['train'].shape[1]}")

print("\n" + "="*60)
print("Cleaning process complete! Useful sensors have been preserved.")



========== [FD001] TASK 1.1 & 1.2: SENSOR AND FEATURE FILTERING ==========
Dropped 7 Constant columns: ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19', 'op_setting_3']
Dropped 2 Near-constant columns (meaningless noise, corr < 0.1): ['op_setting_1', 'op_setting_2']
Successfully KEPT Near-constant columns (with trend):
    -> sensor_6 (Correlation with cycle: 0.1060)
Final number of columns retained (including unit_id, cycle, RUL if any): 18

========== [FD002] TASK 1.1 & 1.2: SENSOR AND FEATURE FILTERING ==========
Dropped 0 Constant columns: []
Dropped 1 Near-constant columns (meaningless noise, corr < 0.1): ['sensor_16']
Final number of columns retained (including unit_id, cycle, RUL if any): 26

========== [FD003] TASK 1.1 & 1.2: SENSOR AND FEATURE FILTERING ==========
Dropped 6 Constant columns: ['sensor_1', 'sensor_5', 'sensor_16', 'sensor_18', 'sensor_19', 'op_setting_3']
Dropped 2 Near-constant columns (meaningless noise, corr < 0.1): ['op_setting_1'

### Task 1.3 - Khởi tạo Nhãn Piecewise RUL.

Gán cho RUL = 125 với các động "mới vận hành" tức là ở các chu kì đầu từ 1 = 50 thì chu kì còn lại vẫn còn lâu, vẫn còn vận hành trơn tru nên chỉ gán RUL = 125 cho các chu kì đầu, khi nào xuống dưới 125 thì mới bắt đầu đếm ngược chính xác

In [852]:
# Cài đặt ngưỡng RUL tối đa (Piecewise RUL)
MAX_RUL = 125
target_fds = TARGET_FDS

for fd in target_fds:
    print(f"\n[{fd}] TASK 1.3: INITIALIZING PIECEWISE RUL LABEL (MAX = {MAX_RUL})")
    train_df = datasets[fd]['train']
    
    # 1. Find the maximum lifetime (max_cycle) for each engine (unit_id)
    max_cycle_df = train_df.groupby('unit_id')['cycle'].max().reset_index()
    max_cycle_df.columns = ['unit_id', 'max_cycle']
    
    # 2. Merge the max_cycle column back into the original training dataframe
    train_df = train_df.merge(max_cycle_df, on='unit_id', how='left')
    
    # 3. Calculate linear RUL (RUL = Total lifetime - Current cycle)
    train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']
    
    # 4. Apply Piecewise technique: Clip RUL at MAX_RUL (125)
    train_df['RUL'] = train_df['RUL'].clip(upper=MAX_RUL)
    
    # 5. Clean up: Remove the intermediate max_cycle column
    train_df.drop(columns=['max_cycle'], inplace=True)
    
    # Update the dictionary
    datasets[fd]['train'] = train_df
    
    print(f"Successfully created RUL label. Train set size: {train_df.shape}")
    
    # Display the first 3 and last 3 cycles of unit_id = 1 for verification
    print(f"Verifying RUL for unit_id = 1:")
    display_df = pd.concat([
        train_df[train_df['unit_id'] == 1].head(3),
        train_df[train_df['unit_id'] == 1].tail(3)
    ])
    print(display_df[['unit_id', 'cycle', 'RUL']].to_string(index=False))

print("\n" + "="*60)
print("Stage 1 Complete! We now have accurate labels for model training.")



[FD001] TASK 1.3: INITIALIZING PIECEWISE RUL LABEL (MAX = 125)
Successfully created RUL label. Train set size: (20631, 18)
Verifying RUL for unit_id = 1:
 unit_id  cycle  RUL
       1      1  125
       1      2  125
       1      3  125
       1    190    2
       1    191    1
       1    192    0

[FD002] TASK 1.3: INITIALIZING PIECEWISE RUL LABEL (MAX = 125)
Successfully created RUL label. Train set size: (53759, 26)
Verifying RUL for unit_id = 1:
 unit_id  cycle  RUL
       1      1  125
       1      2  125
       1      3  125
       1    147    2
       1    148    1
       1    149    0

[FD003] TASK 1.3: INITIALIZING PIECEWISE RUL LABEL (MAX = 125)
Successfully created RUL label. Train set size: (24720, 19)
Verifying RUL for unit_id = 1:
 unit_id  cycle  RUL
       1      1  125
       1      2  125
       1      3  125
       1    257    2
       1    258    1
       1    259    0

[FD004] TASK 1.3: INITIALIZING PIECEWISE RUL LABEL (MAX = 125)
Successfully created RUL label

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd

print(f"STARTING OPERATING CONDITION PROCESSING FOR {target_fds}")

complex_fds = [fd for fd in ['FD002', 'FD004'] if fd in target_fds]
op_cols = ['op_setting_1', 'op_setting_2', 'op_setting_3']
NUM_REGIMES = 6 

for fd in complex_fds:
    print(f"\n{'='*15} CLUSTERING & NORMALIZING {fd} {'='*15}")
    train_df = datasets[fd]['train']
    test_df = datasets[fd]['test']
    
    sensor_cols = [c for c in train_df.columns if 'sensor' in c]
    
    # ---------------------------------------------------------
    # STEP 1: NORMALIZE OP_SETTINGS FOR K-MEANS
    # ---------------------------------------------------------
    op_scaler = StandardScaler()
    train_op_scaled = op_scaler.fit_transform(train_df[op_cols])
    test_op_scaled = op_scaler.transform(test_df[op_cols])

    # ---------------------------------------------------------
    # STEP 2: K-MEANS CLUSTERING ON SCALED DATA
    # ---------------------------------------------------------
    print("1. Performing K-Means clustering for 6 flight regimes...")
    kmeans = KMeans(n_clusters=NUM_REGIMES, random_state=42, n_init=10)
    train_df['regime'] = kmeans.fit_predict(train_op_scaled)
    test_df['regime'] = kmeans.predict(test_op_scaled)
    
    # ---------------------------------------------------------
    # STEP 3: NORMALIZE SENSORS PER REGIME
    # ---------------------------------------------------------
    print("2. Removing environmental noise (Condition-based Normalization)...")
    for regime in range(NUM_REGIMES):
        train_idx = train_df[train_df['regime'] == regime].index
        test_idx = test_df[test_df['regime'] == regime].index
        
        scaler = StandardScaler()
        train_df.loc[train_idx, sensor_cols] = scaler.fit_transform(train_df.loc[train_idx, sensor_cols])
        
        if len(test_idx) > 0:
            test_df.loc[test_idx, sensor_cols] = scaler.transform(test_df.loc[test_idx, sensor_cols])
            
    # Clean up
    train_df.drop(columns=op_cols + ['regime'], inplace=True)
    test_df.drop(columns=op_cols + ['regime'], inplace=True)
    
    datasets[fd]['train'] = train_df
    datasets[fd]['test'] = test_df
    
    print("Completed! Environmental noise has been successfully removed.")


STARTING OPERATING CONDITION PROCESSING FOR FD002 & FD004

=============== CLUSTERING & NORMALIZING FD002 ===============
1. Performing K-Means clustering for 6 flight regimes...
2. Removing environmental noise (Condition-based Normalization)...
Completed! Environmental noise has been successfully removed.

=============== CLUSTERING & NORMALIZING FD004 ===============
1. Performing K-Means clustering for 6 flight regimes...
2. Removing environmental noise (Condition-based Normalization)...
Completed! Environmental noise has been successfully removed.


# Giai đoạn 2: 

Làm mượt tín hiệu (EMA - Exponential Moving Average): Tín hiệu cảm biến thực tế luôn bị rung lắc (nhiễu tần số cao). Làm mượt sẽ giúp mô hình nhìn thấy rõ "đường cong suy thoái" của động cơ hơn.

In [854]:
target_fds = TARGET_FDS
EMA_SPAN = 10

for fd in target_fds:
    print(f"\n{'='*10} [{fd}] STAGE 2 - TASK 2.1: SIGNAL SMOOTHING (EMA) {'='*10}")
    train_df = datasets[fd]['train']
    test_df = datasets[fd]['test']
    
    # Get only sensor columns that passed Stage 1 (ignore unit_id, cycle, RUL)
    sensor_cols = [col for col in train_df.columns if col.startswith('sensor_')]
    
    def apply_ema(df, cols, span):
        df_smoothed = df.copy()
        # Group by unit_id to avoid signal leakage between engines
        for col in cols:
            df_smoothed[col] = df_smoothed.groupby('unit_id')[col].transform(
                lambda x: x.ewm(span=span, adjust=False).mean()
            )
        return df_smoothed

    # Apply smoothing to both Train and Test sets
    datasets[fd]['train'] = apply_ema(train_df, sensor_cols, EMA_SPAN)
    datasets[fd]['test'] = apply_ema(test_df, sensor_cols, EMA_SPAN)
    
    print(f"Applied EMA filter (span={EMA_SPAN}) to smooth {len(sensor_cols)} sensors.")
    
    # Print the first 5 values of the first sensor for verification
    sample_col = sensor_cols[0]
    print(f"Verifying '{sample_col}' for unit_id=1 (Train) after smoothing:")
    print(datasets[fd]['train'][datasets[fd]['train']['unit_id'] == 1][sample_col].head(5).tolist())

print("\n" + "="*60)
print("Stage 2 Complete! Signal noise has been filtered, ready for feature extraction.")



========== [FD001] STAGE 2 - TASK 2.1: SIGNAL SMOOTHING (EMA) ==========
Applied EMA filter (span=10) to smooth 15 sensors.
Verifying 'sensor_2' for unit_id=1 (Train) after smoothing:
[641.82, 641.88, 641.9654545454545, 642.0353719008262, 642.0962133734032]

========== [FD002] STAGE 2 - TASK 2.1: SIGNAL SMOOTHING (EMA) ==========
Applied EMA filter (span=10) to smooth 20 sensors.
Verifying 'sensor_1' for unit_id=1 (Train) after smoothing:
[5.684341886080802e-14, 4.6508251795206554e-14, 3.8052206014259904e-14, 3.1133623102576284e-14, 2.547296435665332e-14]

========== [FD003] STAGE 2 - TASK 2.1: SIGNAL SMOOTHING (EMA) ==========
Applied EMA filter (span=10) to smooth 16 sensors.
Verifying 'sensor_2' for unit_id=1 (Train) after smoothing:
[642.36, 642.3854545454545, 642.3480991735537, 642.4520811419984, 642.3117027525441]

========== [FD004] STAGE 2 - TASK 2.1: SIGNAL SMOOTHING (EMA) ==========
Applied EMA filter (span=10) to smooth 20 sensors.
Verifying 'sensor_1' for unit_id=1 (Train)

# GIAI ĐOẠN 3: Trích xuất Đặc trưng Nâng cao

Với Task 3.1, set window_size = 15.

Với Task 3.2, set shift_size = 10 như gợi ý của bạn. Để xử lý các giá trị NaN ở 10 chu kỳ đầu tiên do hàm .shift() sinh ra, sử dụng lệnh .bfill() (lấy giá trị đầu tiên lấp ngược lên), giúp "độ dốc" ở những chu kỳ đầu được tính bằng giá trị hiện tại trừ đi giá trị khởi điểm.

In [855]:
window_size = 15
shift_size = 10
target_fds = TARGET_FDS

for fd in target_fds:
    print(f"\n{'='*10} [{fd}] STAGE 3: ADVANCED FEATURE EXTRACTION {'='*10}")
    train_df = datasets[fd]['train']
    test_df = datasets[fd]['test']
    
    # Get only the original smoothed sensor columns
    sensor_cols = [c for c in train_df.columns if c.startswith('sensor_')]
    
    def extract_advanced_features(df, cols, w_size, s_size):
        df_feat = df.copy()
        
        for col in cols:
            grouped = df_feat.groupby('unit_id')[col]
            
            # --- TASK 3.1: ROLLING FEATURES (Sliding Window) ---
            # min_periods=1 allows calculation from the first cycle without full NaNs
            df_feat[f'{col}_roll_mean'] = grouped.transform(lambda x: x.rolling(w_size, min_periods=1).mean())
            df_feat[f'{col}_roll_std'] = grouped.transform(lambda x: x.rolling(w_size, min_periods=1).std().fillna(0))
            df_feat[f'{col}_roll_min'] = grouped.transform(lambda x: x.rolling(w_size, min_periods=1).min())
            df_feat[f'{col}_roll_max'] = grouped.transform(lambda x: x.rolling(w_size, min_periods=1).max())
            
            # --- TASK 3.2: TREND / DELTA FEATURES (Dynamic Features) ---
            # Slope = Current value - Value from 10 cycles ago
            # Use bfill() to borrow the first value for the first 10 missing cycles due to shift
            df_feat[f'{col}_trend'] = grouped.transform(lambda x: x - x.shift(s_size).bfill())
            
        return df_feat

    # Execute extraction on both Train and Test sets
    datasets[fd]['train'] = extract_advanced_features(train_df, sensor_cols, window_size, shift_size)
    datasets[fd]['test'] = extract_advanced_features(test_df, sensor_cols, window_size, shift_size)
    
    # Calculate the number of new features created
    num_new_cols = len(sensor_cols) * 5 # Each sensor generates 5 new columns (4 roll + 1 trend)
    print(f"Successfully created {num_new_cols} new features for each dataset.")
    print(f"Current Train set size: {datasets[fd]['train'].shape}")

print("\n" + "="*60)
print("Stage 3 Complete! Your data is now a goldmine of information.")



========== [FD001] STAGE 3: ADVANCED FEATURE EXTRACTION ==========
Successfully created 75 new features for each dataset.
Current Train set size: (20631, 93)

========== [FD002] STAGE 3: ADVANCED FEATURE EXTRACTION ==========
Successfully created 100 new features for each dataset.
Current Train set size: (53759, 123)

========== [FD003] STAGE 3: ADVANCED FEATURE EXTRACTION ==========
Successfully created 80 new features for each dataset.
Current Train set size: (24720, 99)

========== [FD004] STAGE 3: ADVANCED FEATURE EXTRACTION ==========
Successfully created 100 new features for each dataset.
Current Train set size: (61249, 123)

Stage 3 Complete! Your data is now a goldmine of information.


## GIAI ĐOẠN 4: Chuẩn hóa Dữ liệu (Scaling & Normalization)

Task 4.1 (Xử lý đa điều kiện): Với cấu hình `TARGET_FDS = ['FD003', 'FD004']`, bước K-Means chuẩn hóa theo regime sẽ áp dụng cho tập phù hợp trong giai đoạn trước.\n

In [857]:
from sklearn.preprocessing import MinMaxScaler

target_fds = TARGET_FDS

for fd in target_fds:
    print(f"\n{'='*10} [{fd}] STAGE 4: DATA NORMALIZATION {'='*10}")
    train_df = datasets[fd]['train']
    test_df = datasets[fd]['test']
    
    # Identify all columns to be normalized (including original sensors + advanced features)
    # Exclude identifier and target columns
    cols_to_exclude = ['unit_id', 'cycle', 'RUL']
    feature_cols = [col for col in train_df.columns if col not in cols_to_exclude]
    
    # --- TASK 4.2: GLOBAL NORMALIZATION USING MIN-MAX SCALER ---
    scaler = MinMaxScaler()
    
    # Only .fit_transform() on the training set for the model to learn parameters (Min, Max)
    train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    
    # Use .transform() on the test set to apply the same scaling
    test_df[feature_cols] = scaler.transform(test_df[feature_cols])
    
    # Update the dictionary
    datasets[fd]['train'] = train_df
    datasets[fd]['test'] = test_df
    
    print(f"Normalized all {len(feature_cols)} features to the range [0, 1].")
    
    # Print the first 3 values of the first '_roll_std' column for verification
    sample_col = [c for c in feature_cols if '_roll_std' in c][0]
    print(f"Verifying '{sample_col}' for unit_id=1 (Train) after scaling:")
    print(train_df[train_df['unit_id'] == 1][sample_col].head(3).tolist())

print("\n" + "="*60)
print("Stage 4 Complete! Data is now fully balanced and ready for modeling.")



========== [FD001] STAGE 4: DATA NORMALIZATION ==========
Normalized all 90 features to the range [0, 1].
Verifying 'sensor_2_roll_std' for unit_id=1 (Train) after scaling:
[0.0, 0.1791278607581686, 0.3086239760222294]

========== [FD002] STAGE 4: DATA NORMALIZATION ==========
Normalized all 120 features to the range [0, 1].
Verifying 'sensor_1_roll_std' for unit_id=1 (Train) after scaling:
[0.0, 0.08554306492546147, 0.11016147434472552]

========== [FD003] STAGE 4: DATA NORMALIZATION ==========
Normalized all 96 features to the range [0, 1].
Verifying 'sensor_2_roll_std' for unit_id=1 (Train) after scaling:
[0.0, 0.05879943778518087, 0.06234069862465485]

========== [FD004] STAGE 4: DATA NORMALIZATION ==========
Normalized all 120 features to the range [0, 1].
Verifying 'sensor_1_roll_std' for unit_id=1 (Train) after scaling:
[0.0, 0.2943611532668803, 0.22174855242127045]

Stage 4 Complete! Data is now fully balanced and ready for modeling.


# GIAI ĐOẠN 5: Định hình Dữ liệu đầu vào (Data Shaping)

Task 5.1 - Giữ lại toàn bộ raw sensor cùng các cột feature đã tạo ở Giai đoạn 3. Không drop raw sensors hay op_setting ở bước này để mô hình có thể khai thác cả tín hiệu gốc lẫn tín hiệu đã được engineering.

Task 5.2 - Lấy mẫu tập Test (Cho việc Evaluate): Tập Test chỉ dùng để đánh giá RUL ở thời điểm hiện tại. Lọc ra dòng cuối cùng (.groupby('unit_id').last()) của mỗi động cơ trong tập Test.

In [858]:
import numpy as np

target_fds = TARGET_FDS

for fd in target_fds:
    print(f"\n{'='*10} [{fd}] STAGE 5: INPUT DATA SHAPING {'='*10}")
    train_df = datasets[fd]['train'].copy()
    test_df = datasets[fd]['test'].copy()

    # ==========================================
    # TASK 5.1: KEEP RAW SENSORS + ENGINEERED FEATURES
    # ==========================================
    # Skip final cleanup so downstream models can use both raw sensors
    # and engineered features created in Stage 3.
    print(f"[Task 5.1] Skipping raw sensor cleanup. Columns retained (Train): {train_df.shape[1]}")

    # ==========================================
    # TASK 5.2: SAMPLE TEST SET (For Evaluation)
    # ==========================================
    # Predict RUL at the current moment => get the last cycle of each engine
    test_df_last = test_df.groupby('unit_id').last().reset_index()
    print(f"[Task 5.2] Sampled last row of Test set. Number of Test samples (engines): {test_df_last.shape[0]}")

    # Update data
    datasets[fd]['train'] = train_df
    datasets[fd]['test'] = test_df            # Synchronize to prevent subsequent cells from using the old version
    datasets[fd]['test_last'] = test_df_last  # 2D version for classical models
    datasets[fd]['test_full'] = test_df       # Full version for sequence models


# ==========================================
# TASK 5.3: 3D SEQUENCE CUTTING FUNCTION (For LSTM)
# ==========================================
def gen_sequence(id_df, seq_length, seq_cols):
    """
    Cuts a 2D dataframe into 3D sequences (Sliding Window) for each unit_id.
    Output: Tensor with shape (num_samples, seq_length, num_features)
    """
    data_array = id_df[seq_cols].values
    num_elements = data_array.shape[0]

    if num_elements < seq_length:
        return np.empty((0, seq_length, len(seq_cols)))

    for start, stop in zip(range(0, num_elements - seq_length + 1), range(seq_length, num_elements + 1)):
        yield data_array[start:stop, :]

print("\n" + "="*60)
print("COMPLETED ALL 5 STAGES OF DATA PREPROCESSING!")



========== [FD001] STAGE 5: INPUT DATA SHAPING ==========
[Task 5.1] Skipping raw sensor cleanup. Columns retained (Train): 93
[Task 5.2] Sampled last row of Test set. Number of Test samples (engines): 100

========== [FD002] STAGE 5: INPUT DATA SHAPING ==========
[Task 5.1] Skipping raw sensor cleanup. Columns retained (Train): 123
[Task 5.2] Sampled last row of Test set. Number of Test samples (engines): 259

========== [FD003] STAGE 5: INPUT DATA SHAPING ==========
[Task 5.1] Skipping raw sensor cleanup. Columns retained (Train): 99
[Task 5.2] Sampled last row of Test set. Number of Test samples (engines): 100

========== [FD004] STAGE 5: INPUT DATA SHAPING ==========
[Task 5.1] Skipping raw sensor cleanup. Columns retained (Train): 123
[Task 5.2] Sampled last row of Test set. Number of Test samples (engines): 248

COMPLETED ALL 5 STAGES OF DATA PREPROCESSING!


In [859]:
import numpy as np
import pandas as pd

print("STARTING TRAIN/TEST MATRIX SYNCHRONIZATION FOR RF AND XGBOOST\n")

target_fds = TARGET_FDS

for fd in target_fds:
    print(f"========== PREPARING X_train / y_train / X_test / y_test FOR {fd} ==========")

    train_df = datasets[fd]['train'].copy()
    test_last_df = datasets[fd]['test_last'].copy()

    # Step 1: Separate target from train
    X_train_raw = train_df.drop(columns=['RUL'], errors='ignore').copy()
    y_train = train_df['RUL'].astype(float).copy()
    X_test_raw = test_last_df.copy()

    # Step 2: Standardize y_test by unit_id to avoid order mismatch
    rul_df = datasets[fd]['rul'][['RUL']].copy().reset_index(drop=True)
    rul_df['unit_id'] = np.arange(1, len(rul_df) + 1)

    # Step 3: Separate groups for GroupKFold
    if 'unit_id' in X_train_raw.columns:
        groups = X_train_raw['unit_id'].copy().astype(int)
    else:
        groups = datasets[fd]['train']['unit_id'].copy().astype(int)

    if 'unit_id' in X_test_raw.columns:
        test_unit_ids = X_test_raw['unit_id'].copy().astype(int)
    else:
        test_unit_ids = pd.Series(np.arange(1, len(X_test_raw) + 1), index=X_test_raw.index)

    # Step 4: Remove identifier columns from model input
    X_train = X_train_raw.drop(columns=['unit_id', 'cycle'], errors='ignore').copy()
    X_test = X_test_raw.drop(columns=['unit_id', 'cycle'], errors='ignore').copy()

    # Remove duplicate columns and sanitize column names for XGBoost safety
    X_train = X_train.loc[:, ~X_train.columns.duplicated()].copy()
    X_test = X_test.loc[:, ~X_test.columns.duplicated()].copy()
    X_train.columns = X_train.columns.astype(str).str.replace(r'[\[\]<>]', '_', regex=True)
    X_test.columns = X_test.columns.astype(str).str.replace(r'[\[\]<>]', '_', regex=True)

    # Synchronize test schema with train schema
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0.0)

    # Step 5: Create y_test in the correct order of unit_id in X_test
    y_test = (
        pd.DataFrame({'unit_id': test_unit_ids.values})
        .merge(rul_df[['unit_id', 'RUL']], on='unit_id', how='left')['RUL']
        .astype(float)
        .values
    )

    if np.isnan(y_test).any():
        raise ValueError(f"{fd}: Some unit_id in X_test could not be mapped to y_test.")

    datasets[fd]['model_data'] = {
        'X_train': X_train,
        'y_train': y_train,
        'X_test': X_test,
        'y_test': y_test,
        'groups': groups,
        'feature_cols': X_train.columns.tolist()
    }

    print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
    print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")

print("\nCOMPLETE! Data is synchronized and ready for tuning for both models.")


STARTING TRAIN/TEST MATRIX SYNCHRONIZATION FOR RF AND XGBOOST

========== PREPARING X_train / y_train / X_test / y_test FOR FD001 ==========
X_train: (20631, 90) | X_test: (100, 90)
y_train: (20631,) | y_test: (100,)

COMPLETE! Data is synchronized and ready for tuning for both models.


In [860]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import mean_squared_error
import time

print("TUNING XGBOOST WITH GROUP-K-FOLD (DEEPER SEARCH)\n")

# Expand hyperparameter space for the model to learn more thoroughly and balance bias-variance better
xgb_param_grid = {
    'n_estimators': [300, 500, 800, 1200],
    'learning_rate': [0.005, 0.01, 0.03, 0.05],
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 3, 5, 7, 10],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.3, 1, 3, 5],
    'reg_alpha': [0, 0.01, 0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 1.5, 2.0, 3.0]
}

tuned_xgb_results = {}

for fd in target_fds:
    print(f"========== [{fd}] TUNING XGBOOST ==========")

    try:
        md = datasets[fd]['model_data']
        X_train = md['X_train']
        y_train = md['y_train']
        X_test = md['X_test']
        y_test = md['y_test']
        groups = md['groups']

        xgb_base = XGBRegressor(
            random_state=42,
            n_jobs=1,
            objective='reg:squarederror',
            eval_metric='rmse',
            tree_method='hist',
            verbosity=0
        )

        # Increase the number of folds for a more stable generalization estimate
        gkf = GroupKFold(n_splits=5)

        xgb_random = RandomizedSearchCV(
            estimator=xgb_base,
            param_distributions=xgb_param_grid,
            n_iter=60,
            cv=gkf,
            scoring='neg_root_mean_squared_error',
            verbose=1,
            random_state=42,
            n_jobs=-1
        )

        start_time = time.time()
        xgb_random.fit(X_train, y_train, groups=groups)
        elapsed = time.time() - start_time
        print(f"XGBoost tuning complete in {elapsed:.2f} seconds!")

        best_xgb_params = xgb_random.best_params_
        print(f"Best params (XGB):\n   {best_xgb_params}")
        print(f"   Best CV RMSE: {-xgb_random.best_score_:.4f}")

        datasets[fd]['best_params'] = {'xgb': best_xgb_params}

        # ==========================================
        # TRAIN XGBOOST WITH BEST PARAMS ON THE ENTIRE TRAIN SET
        # ==========================================
        print(f"\nTRAINING XGBOOST ON THE ENTIRE TRAIN SET...\n")
        
        best_xgb_model = XGBRegressor(
            random_state=42,
            n_jobs=-1,
            objective='reg:squarederror',
            eval_metric='rmse',
            **best_xgb_params
        )
        
        best_xgb_model.fit(X_train, y_train)
        
        # Evaluate
        y_train_pred = best_xgb_model.predict(X_train)
        y_test_pred = best_xgb_model.predict(X_test)
        
        rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
        rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
        
        print(f"XGBOOST Performance:")
        print(f"   Train RMSE: {rmse_train:.3f}")
        print(f"   Test RMSE: {rmse_test:.3f}")
        
        # Feature importance
        importances = best_xgb_model.feature_importances_
        feat_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances})
        feat_df = feat_df.sort_values(by='Importance', ascending=False).head(5)
        
        print("\nTOP 5 FEATURES (XGB):")
        for _, row in feat_df.iterrows():
            print(f"   {row['Feature']}: {row['Importance']*100:.2f}%")
        
        # Save model
        datasets[fd]['trained_models'] = {'xgb': best_xgb_model}
        datasets[fd]['evaluation_results'] = {'xgb_train_rmse': rmse_train, 'xgb_test_rmse': rmse_test}

        print("-" * 60 + "\n")

    except Exception as e:
        print(f"ERROR in set {fd}: {e}\n")

print("COMPLETED TUNING + TRAINING XGBOOST!")


TUNING XGBOOST WITH GROUP-K-FOLD (DEEPER SEARCH)

========== [FD001] TUNING XGBOOST ==========
Fitting 5 folds for each of 60 candidates, totalling 300 fits
XGBoost tuning complete in 958.12 seconds!
Best params (XGB):
   {'subsample': 0.8, 'reg_lambda': 1.5, 'reg_alpha': 1.0, 'n_estimators': 1200, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 0.8}
   Best CV RMSE: 14.6982

TRAINING XGBOOST ON THE ENTIRE TRAIN SET...

XGBOOST Performance:
   Train RMSE: 9.983
   Test RMSE: 15.005

TOP 5 FEATURES (XGB):
   sensor_4: 32.07%
   sensor_15: 16.09%
   sensor_11: 7.49%
   sensor_2: 7.11%
   sensor_3: 5.29%
------------------------------------------------------------

COMPLETED TUNING + TRAINING XGBOOST!


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import mean_squared_error
import time

print("TUNING RANDOM FOREST WITH GROUP-K-FOLD\n")

rf_param_grid = {
    'n_estimators': [100, 150, 200], # Slightly increase the number of trees (Increases stability, learns more perspectives)
    'max_depth': [7, 10, 12],        # Allow trees to grow a bit deeper (was 9, now up to 12) to learn complex patterns from 90+ columns
    'min_samples_split': [2, 5, 8],  # Slightly relax to make splitting easier
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 0.5]    # Reduce from 0.7 to 0.5 (Use only 50% of columns at each split) to COMPENSATE for the increased max_depth
}
tuned_rf_results = {}

for fd in target_fds:
    print(f"========== [{fd}] TUNING RANDOM FOREST ==========")

    try:
        md = datasets[fd]['model_data']
        X_train = md['X_train']
        y_train = md['y_train']
        X_test = md['X_test']
        y_test = md['y_test']
        groups = md['groups']

        rf_base = RandomForestRegressor(
            random_state=42,
            n_jobs=1
        )

        gkf = GroupKFold(n_splits=3)

        rf_random = RandomizedSearchCV(
            estimator=rf_base,
            param_distributions=rf_param_grid,
            n_iter=10,
            cv=gkf,
            scoring='neg_root_mean_squared_error',
            verbose=1,
            random_state=42,
            n_jobs=-1
        )

        start_time = time.time()
        rf_random.fit(X_train, y_train, groups=groups)
        elapsed = time.time() - start_time
        print(f"Random Forest tuning complete in {elapsed:.2f} seconds!")

        best_rf_params = rf_random.best_params_
        print(f"Best params (RF):\n   {best_rf_params}")
        print(f"   Best CV RMSE: {-rf_random.best_score_:.4f}")

        # Update best_params dict (keeping XGB from the previous cell)
        if 'best_params' not in datasets[fd]:
            datasets[fd]['best_params'] = {}
        datasets[fd]['best_params']['rf'] = best_rf_params

        # ==========================================
        # TRAIN RANDOM FOREST WITH BEST PARAMS ON THE ENTIRE TRAIN SET
        # ==========================================
        print(f"\nTRAINING RANDOM FOREST ON THE ENTIRE TRAIN SET...\n")
        
        best_rf_model = RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
            **best_rf_params
        )
        
        best_rf_model.fit(X_train, y_train)
        
        # Evaluate
        y_train_pred = best_rf_model.predict(X_train)
        y_test_pred = best_rf_model.predict(X_test)
        
        rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
        rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
        
        print(f"RANDOM FOREST Performance:")
        print(f"   Train RMSE: {rmse_train:.3f}")
        print(f"   Test RMSE: {rmse_test:.3f}")
        
        # Feature importance
        importances = best_rf_model.feature_importances_
        feat_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances})
        feat_df = feat_df.sort_values(by='Importance', ascending=False).head(5)
        
        print("\nTOP 5 FEATURES (RF):")
        for _, row in feat_df.iterrows():
            print(f"   {row['Feature']}: {row['Importance']*100:.2f}%")
        
        # Update trained_models dict (keeping XGB from the previous cell)
        if 'trained_models' not in datasets[fd]:
            datasets[fd]['trained_models'] = {}
        datasets[fd]['trained_models']['rf'] = best_rf_model
        
        # Update evaluation_results
        if 'evaluation_results' not in datasets[fd]:
            datasets[fd]['evaluation_results'] = {}
        datasets[fd]['evaluation_results']['rf_train_rmse'] = rmse_train
        datasets[fd]['evaluation_results']['rf_test_rmse'] = rmse_test

        print("-" * 60 + "\n")

    except Exception as e:
        print(f"ERROR in set {fd}: {e}\n")

print("COMPLETED TUNING + TRAINING RANDOM FOREST!")


TUNING RANDOM FOREST WITH GROUP-K-FOLD

========== [FD001] TUNING RANDOM FOREST ==========
Fitting 3 folds for each of 10 candidates, totalling 30 fits


In [ ]:
import numpy as np
import pandas as pd

print("SUMMARY OF TUNING + TRAINING RESULTS\n")

for fd in target_fds:
    print(f"========== [{fd}] ==========")
    
    if 'evaluation_results' not in datasets[fd]:
        print(f"No results yet. Please run Cells 73 & 74 first.\n")
        continue
    
    res = datasets[fd]['evaluation_results']
    trained = datasets[fd]['trained_models']
    
    print(f"RF   - Train RMSE: {res['rf_train_rmse']:.3f} | Test RMSE: {res['rf_test_rmse']:.3f}")
    print(f"XGB  - Train RMSE: {res['xgb_train_rmse']:.3f} | Test RMSE: {res['xgb_test_rmse']:.3f}")
    
    # Print top 5 feature importance from trained models
    md = datasets[fd]['model_data']
    X_train = md['X_train']
    
    rf_imp = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': trained['rf'].feature_importances_
    }).sort_values('Importance', ascending=False).head(5)
    
    xgb_imp = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': trained['xgb'].feature_importances_
    }).sort_values('Importance', ascending=False).head(5)
    
    print("\nTOP 5 FEATURE IMPORTANCE - RANDOM FOREST")
    for _, row in rf_imp.iterrows():
        print(f"   {row['Feature']}: {row['Importance']*100:.2f}%")
    
    print("\nTOP 5 FEATURE IMPORTANCE - XGBOOST")
    for _, row in xgb_imp.iterrows():
        print(f"   {row['Feature']}: {row['Importance']*100:.2f}%")
    
    print("-" * 60 + "\n")

print("COMPLETE! Both models are ready for Evaluation & Visualization.")


SUMMARY OF TUNING + TRAINING RESULTS

========== [FD001] ==========
RF   - Train RMSE: 5.677 | Test RMSE: 15.252


KeyError: 'xgb_train_rmse'

In [ ]:
import pandas as pd

print("RMSE SUMMARY AFTER TUNING + FINAL TRAINING (RF vs XGB)")

summary_rows = []
for fd in target_fds:
    if 'evaluation_results' not in datasets[fd]:
        continue
    res = datasets[fd]['evaluation_results']
    summary_rows.append({
        'Dataset': fd,
        'RF Train RMSE': round(res['rf_train_rmse'], 3),
        'RF Test RMSE': round(res['rf_test_rmse'], 3),
        'XGB Train RMSE': round(res['xgb_train_rmse'], 3),
        'XGB Test RMSE': round(res['xgb_test_rmse'], 3)
    })

summary_df = pd.DataFrame(summary_rows)

if summary_df.empty:
    print("No evaluation results found. Please run the final train cell first.")
else:
    display(summary_df)
    print("\nConsistent pipeline: Data synchronization -> RF+XGB Tuning -> Final Training -> Evaluation")


📌 TÓM TẮT RMSE SAU TUNING + TRAIN FINAL (RF vs XGB)


,Dataset,RF Train RMSE,RF Test RMSE,XGB Train RMSE,XGB Test RMSE
0,FD001,14.477,17.665,12.756,17.901



✅ Pipeline nhất quán: Đồng bộ dữ liệu -> Tuning RF+XGB -> Train final -> Đánh giá
